# 🔍 Notebook 3: Deduplication

Why store the same file twice? Imagine 10 million users all upload the same viral meme — without
deduplication, your storage bill explodes. In this notebook, we’ll explore how a Dropbox-like system
avoids redundant storage using **deduplication** (dedup). You’ll implement file-level and chunk-level
dedup, build a content-defined chunker, and measure exactly how much space you save.

## Learning Objectives

By the end of this notebook, you’ll understand:
- Why deduplication is critical for cloud storage systems
- How **file-level dedup** uses SHA-256 fingerprints to avoid duplicate uploads
- How **chunk-level dedup** saves space even when files aren’t identical
- The difference between **fixed-size chunking** and **content-defined chunking (CDC)**
- How to calculate and visualize storage savings
- How **reference counting** prevents premature deletion of shared data

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/dropbox
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): [http://localhost:8080](http://localhost:8080)  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `dropbox_demo`
- **MinIO Console** (S3 GUI): [http://localhost:9001](http://localhost:9001)  
  Login: User `minioadmin`, Password `minioadmin`
- **RedisInsight** (Redis GUI): [http://localhost:5540](http://localhost:5540)  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code’s kernel picker (top-right of notebook).  
If it doesn’t appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import boto3
import hashlib
import os

# ── Connection settings ─────────────────────────────────────────
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'dropbox_demo',
    'user': 'demo',
    'password': 'demo',
}

REDIS_CONFIG = {
    'host': 'localhost',
    'port': 6379,
    'decode_responses': True,
}

S3_CONFIG = {
    'endpoint_url': 'http://localhost:9000',
    'aws_access_key_id': 'minioadmin',
    'aws_secret_access_key': 'minioadmin',
}

BUCKET = 'dropbox-files'

# ── Helper functions ──────────────────────────────────────────
def get_db():
    """Return a Postgres connection with autocommit enabled."""
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)

def get_s3():
    """Return a boto3 S3 client pointed at MinIO."""
    return boto3.client('s3', **S3_CONFIG)

def sha256_hex(data: bytes) -> str:
    """Compute SHA-256 hash of bytes, return as a 64-char hex string."""
    return hashlib.sha256(data).hexdigest()

# ── Test connections ───────────────────────────────────────────
try:
    conn = get_db()
    conn.close()
    print('✅ Connected to PostgreSQL')
except Exception as e:
    print(f'❌ PostgreSQL failed: {e}')
    print('   Run: cd system-designs/dropbox && docker-compose up -d')

try:
    r = get_redis()
    r.ping()
    print('✅ Connected to Redis')
except Exception as e:
    print(f'❌ Redis failed: {e}')

try:
    s3 = get_s3()
    s3.head_bucket(Bucket=BUCKET)
    print('✅ Connected to MinIO (S3)')
except Exception as e:
    print(f'❌ MinIO failed: {e}')

## 💡 Why Deduplication Matters

### The Problem

Think about how people use cloud storage:

- A company sends the same PDF to 500 employees — all 500 upload it to their Dropbox.
- A meme goes viral — millions of users save the same image.
- You edit one slide in a 50 MB presentation — do you really need to re-upload all 50 MB?

Without deduplication, each upload creates a **separate copy** in storage. If a 5 MB file is
uploaded by 1,000 users, that’s **5 GB** wasted on identical data.

### The Library Analogy 📚

A library doesn’t buy a new copy of *Harry Potter* for every reader who wants it.
It buys **one copy** and lends it to many people. Deduplication works the same
way — store the data **once**, and let multiple users **reference** it.

### Real-World Impact

Dropbox has reported that deduplication saves over **25% of total storage**. At
their scale (billions of files), that translates to **petabytes** of savings and
millions of dollars in storage costs.

Let’s see exactly how much waste there is without dedup:

In [ ]:
# === The waste without deduplication ===

file_size_mb = 5        # A 5 MB file
num_users = 1000        # 1,000 users upload the same file

without_dedup_mb = file_size_mb * num_users
with_dedup_mb = file_size_mb  # Just one copy!
savings_mb = without_dedup_mb - with_dedup_mb
savings_pct = (savings_mb / without_dedup_mb) * 100

print('\U0001f4ca Deduplication savings for ONE popular file:')
print(f'   Without dedup: {without_dedup_mb:,} MB ({num_users} copies)')
print(f'   With dedup:    {with_dedup_mb:,} MB (1 copy)')
print(f'   Saved:         {savings_mb:,} MB ({savings_pct:.1f}%)')
print()
print('Now imagine this across millions of files...')
print(f'   If 30% of files are duplicates across 100 TB of data:')
print(f'   Savings = {100 * 0.30:.0f} TB = 30,000 GB saved! \U0001f4b0')

## 🔐 File-Level Deduplication

The simplest form of dedup: **hash the entire file** and check if that hash already exists.

### How It Works

1. **Compute a fingerprint** — Hash the file’s bytes with SHA-256 → a 64-character hex string.
2. **Check the database** — Does any existing file have this fingerprint?
3. **If YES** → Skip the upload! Create a new metadata record pointing to the *same* storage location.
4. **If NO** → Upload the file to MinIO, save the metadata with the fingerprint.

```
Alice uploads "report.pdf"  (SHA-256: abc123...)
  → No match in DB → Upload to MinIO → Save metadata

Bob uploads "report.pdf"  (SHA-256: abc123...)
  → Match found! → Skip upload → Save metadata pointing to same storage key
```

Let’s see this in action:

In [ ]:
# === Step 1: Clean slate ===
conn = get_db()
cur = conn.cursor()
cur.execute('DELETE FROM chunks')
cur.execute('DELETE FROM files')

# Clean MinIO too
s3 = get_s3()
response = s3.list_objects_v2(Bucket=BUCKET)
if 'Contents' in response:
    for obj in response['Contents']:
        s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
print('\U0001f9f9 Starting fresh \u2014 cleared old demo data')
print()

# === Step 2: Alice uploads "company_report.pdf" ===
file_content = b'Q1 2024 Revenue Report - Confidential - ' * 1000  # ~40 KB
file_name = 'company_report.pdf'
fingerprint = sha256_hex(file_content)

print(f'\U0001f4c4 Alice wants to upload: {file_name}')
print(f'   Size: {len(file_content):,} bytes')
print(f'   SHA-256: {fingerprint[:16]}...')

# Check for duplicates
cur.execute(
    'SELECT id, storage_key FROM files WHERE fingerprint = %s AND status = %s LIMIT 1',
    (fingerprint, 'uploaded'),
)
existing = cur.fetchone()

if existing:
    storage_key = existing[1]
    print(f'   \U0001f501 Duplicate found! Reusing storage from file {existing[0]}')
else:
    # First time \u2014 upload to MinIO
    storage_key = f'files/{fingerprint}'
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=file_content)
    print(f'   \U0001f4e4 Uploaded to MinIO: {storage_key}')

# Save metadata record for Alice (owner_id=1)
cur.execute(
    '''INSERT INTO files (owner_id, file_name, mime_type, file_size, fingerprint, storage_key, status)
    VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id''',
    (1, file_name, 'application/pdf', len(file_content), fingerprint, storage_key, 'uploaded'),
)
alice_file_id = cur.fetchone()[0]
print(f'   \u2705 File record created for Alice (file_id={alice_file_id})')
conn.close()

In [ ]:
# === Step 3: Bob uploads the EXACT same file ===
bob_content = b'Q1 2024 Revenue Report - Confidential - ' * 1000  # Same bytes!
bob_fingerprint = sha256_hex(bob_content)

print(f'\U0001f4c4 Bob wants to upload: {file_name}')
print(f'   SHA-256: {bob_fingerprint[:16]}...')
print(f'   Same as Alice? {bob_fingerprint == fingerprint}')
print()

conn = get_db()
cur = conn.cursor()

# Check for duplicates \u2014 this time we WILL find one!
cur.execute(
    'SELECT id, storage_key FROM files WHERE fingerprint = %s AND status = %s LIMIT 1',
    (bob_fingerprint, 'uploaded'),
)
existing = cur.fetchone()

if existing:
    storage_key = existing[1]
    print('\U0001f501 DUPLICATE DETECTED!')
    print(f'   File ID {existing[0]} already has this content.')
    print(f'   Reusing storage key: {storage_key}')
    print(f'   \u23ed\ufe0f  Zero bytes uploaded to MinIO!')
else:
    storage_key = f'files/{bob_fingerprint}'
    s3 = get_s3()
    s3.put_object(Bucket=BUCKET, Key=storage_key, Body=bob_content)
    print('   \U0001f4e4 Uploaded to MinIO')

# Save Bob's metadata (points to the same storage!)
cur.execute(
    '''INSERT INTO files (owner_id, file_name, mime_type, file_size, fingerprint, storage_key, status)
    VALUES (%s, %s, %s, %s, %s, %s, %s) RETURNING id''',
    (2, 'company_report.pdf', 'application/pdf', len(bob_content), bob_fingerprint, storage_key, 'uploaded'),
)
bob_file_id = cur.fetchone()[0]
print(f'   \u2705 File record created for Bob (file_id={bob_file_id})')
conn.close()

In [ ]:
# === Step 4: Verify \u2014 two records, one stored copy ===
conn = get_db()
cur = conn.cursor()

cur.execute(
    'SELECT id, owner_id, file_name, file_size, storage_key FROM files WHERE fingerprint = %s',
    (fingerprint,),
)
rows = cur.fetchall()

print(f'\U0001f4ca Database: {len(rows)} file records with the same fingerprint')
print('-' * 70)
for row in rows:
    owner_name = {1: 'alice', 2: 'bob', 3: 'charlie'}.get(row[1], f'user_{row[1]}')
    print(f'   File ID {row[0]:>3} | owner: {owner_name:<8} | size: {row[3]:>10,} bytes')
    print(f'              | storage_key: {row[4]}')

# Count actual objects in MinIO
s3 = get_s3()
objects = s3.list_objects_v2(Bucket=BUCKET)
obj_count = objects.get('KeyCount', 0)

print(f'\n\U0001f4be MinIO: {obj_count} object(s) in bucket')
print(f'\n\U0001f389 Result: {len(rows)} users own the file, but only {obj_count} copy is stored!')
print(f'   Storage saved: {len(file_content):,} bytes (one full copy avoided)')
conn.close()

## 🧩 Chunk-Level Deduplication

File-level dedup is great when files are **exactly identical**. But what about files that
are *mostly* the same?

### The Problem

Imagine a 50 MB PowerPoint presentation. You change one slide and re-upload.
File-level dedup sees a *different* SHA-256 (because even one changed byte produces
a completely different hash), so it stores the entire 50 MB again.

### The Solution: Break Files into Chunks

Instead of hashing the whole file, we:
1. **Split the file into chunks** (e.g., 8 KB each)
2. **Hash each chunk independently**
3. **Check each chunk’s fingerprint** against the database
4. **Only upload chunks that are new**

This way, if two files share 80% of their content, only the 20% that’s different
gets uploaded.

```
File A:  [chunk1] [chunk2] [chunk3] [chunk4] [chunk5]

File B:  [chunk1] [chunk2] [chunk3] [chunk4] [chunk6]  ← only chunk6 is new!
                                               ↑
                                    4 out of 5 chunks reused
                                    Only 1 chunk uploaded = 80% savings
```

Let’s implement this:

In [ ]:
# === Chunk-Level Dedup Implementation ===

CHUNK_SIZE = 8192  # 8 KB per chunk

def split_into_chunks(data: bytes, chunk_size: int = CHUNK_SIZE) -> list:
    """Split data into fixed-size chunks."""
    return [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]


def upload_file_chunked(owner_id, file_name, file_content):
    """Upload a file using chunk-level deduplication.

    For each chunk:
      - Compute its SHA-256 fingerprint
      - If the fingerprint already exists in the DB \u2192 reuse it (skip upload)
      - If it's new \u2192 upload to MinIO

    Returns (file_id, chunks_uploaded, chunks_reused, total_chunks).
    """
    conn = get_db()
    cur = conn.cursor()
    s3 = get_s3()

    file_fingerprint = sha256_hex(file_content)

    # Create the file record
    cur.execute(
        '''INSERT INTO files (owner_id, file_name, file_size, fingerprint, status)
        VALUES (%s, %s, %s, %s, %s) RETURNING id''',
        (owner_id, file_name, len(file_content), file_fingerprint, 'uploading'),
    )
    file_id = cur.fetchone()[0]

    # Split into chunks and process each one
    chunks = split_into_chunks(file_content)
    uploaded = 0
    reused = 0

    for idx, chunk_data in enumerate(chunks):
        chunk_fp = sha256_hex(chunk_data)

        # Check if this chunk fingerprint already exists in storage
        cur.execute(
            'SELECT storage_key FROM chunks WHERE fingerprint = %s AND status = %s LIMIT 1',
            (chunk_fp, 'uploaded'),
        )
        existing = cur.fetchone()

        if existing:
            # Reuse the existing chunk \u2014 no upload needed!
            storage_key = existing[0]
            reused += 1
        else:
            # New chunk \u2014 upload to MinIO
            storage_key = f'chunks/{chunk_fp}'
            s3.put_object(Bucket=BUCKET, Key=storage_key, Body=chunk_data)
            uploaded += 1

        # Record this chunk in the database
        cur.execute(
            '''INSERT INTO chunks (file_id, chunk_index, chunk_size, fingerprint, storage_key, status, uploaded_at)
            VALUES (%s, %s, %s, %s, %s, %s, NOW())''',
            (file_id, idx, len(chunk_data), chunk_fp, storage_key, 'uploaded'),
        )

    # Mark file as uploaded
    cur.execute("UPDATE files SET status = 'uploaded' WHERE id = %s", (file_id,))
    conn.close()

    return file_id, uploaded, reused, len(chunks)

print('\u2705 upload_file_chunked() ready to use')

In [ ]:
# === Create two files that share 80% of their content ===

# 4 shared chunks (32 KB) + 1 unique chunk each (8 KB) = 5 chunks per file
shared_data = os.urandom(CHUNK_SIZE * 4)   # 4 chunks both files share
unique_a    = os.urandom(CHUNK_SIZE)        # 1 chunk unique to File A
unique_b    = os.urandom(CHUNK_SIZE)        # 1 chunk unique to File B

file_a_content = shared_data + unique_a     # 40 KB total
file_b_content = shared_data + unique_b     # 40 KB total

print(f'File A: {len(file_a_content):,} bytes ({len(file_a_content) // CHUNK_SIZE} chunks)')
print(f'File B: {len(file_b_content):,} bytes ({len(file_b_content) // CHUNK_SIZE} chunks)')
print(f'Shared: {len(shared_data):,} bytes ({len(shared_data) // CHUNK_SIZE} chunks)')
print()

# Upload File A (Alice, owner_id=1)
file_a_id, up_a, re_a, total_a = upload_file_chunked(1, 'project_v1.zip', file_a_content)
print(f'\U0001f4e4 File A (Alice): {total_a} chunks \u2192 {up_a} uploaded, {re_a} reused')

# Upload File B (Bob, owner_id=2) \u2014 should reuse 4 out of 5 chunks!
file_b_id, up_b, re_b, total_b = upload_file_chunked(2, 'project_v2.zip', file_b_content)
print(f'\U0001f4e4 File B (Bob):   {total_b} chunks \u2192 {up_b} uploaded, {re_b} reused')

print()
total = total_a + total_b
stored = up_a + up_b
deduped = re_a + re_b
print('\U0001f389 Chunk-level dedup results:')
print(f'   Total chunks across both files: {total}')
print(f'   Chunks actually stored in MinIO: {stored}')
print(f'   Chunks reused (deduped):         {deduped}')
print(f'   Storage saved: {deduped * CHUNK_SIZE:,} bytes ({deduped / total * 100:.0f}%)')

## ✂️ Content-Defined Chunking (CDC) vs Fixed-Size Chunking

### The Problem with Fixed-Size Chunks

What happens when you insert a few bytes at the **beginning** of a file?

```
Original:    [  chunk 1  ] [  chunk 2  ] [  chunk 3  ] [  chunk 4  ]

After edit:  [XX chunk 1 ] [ chunk 2 ..] [.. chunk 3 ] [. chunk 4 .]
                ↑ shifted!    ↑ shifted!     ↑ shifted!    ↑ shifted!
```

**Every single chunk boundary shifts!** All chunks now have different content,
so the fingerprints change and **zero chunks match** — dedup saves nothing.

### The Solution: Content-Defined Chunking (CDC)

Instead of cutting at fixed positions (every 8 KB), CDC picks boundaries
**based on the content itself**.

**How it works:**
1. Slide a window over the data, computing a **rolling hash** (like a Rabin fingerprint)
2. When the hash meets a condition (e.g., `hash % avg_size == 0`), create a boundary
3. Boundaries depend on **local content**, not global position

**Why this helps:** Inserting bytes at the start only affects the *first* chunk.
All later boundaries are still determined by the same content, so they stay put:

```
Original:    [chunk 1] [  chunk 2  ] [   chunk 3   ] [chunk 4]

After edit:  [XX chunk 1..] [  chunk 2  ] [   chunk 3   ] [chunk 4]
                ↑ changed      ↑ same!         ↑ same!       ↑ same!
```

Let’s implement both and compare:

In [ ]:
def fixed_size_chunks(data: bytes, chunk_size: int = 8192) -> list:
    """Split data into fixed-size chunks."""
    return [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]


def cdc_chunks(data: bytes, min_size=2048, max_size=16384, avg_size=8192) -> list:
    """Content-Defined Chunking using a simplified rolling hash.

    How it works:
    - We slide through the data one byte at a time
    - At each position, we update a rolling hash
    - When (hash & mask) == 0, we declare a chunk boundary
    - min_size prevents tiny chunks; max_size caps huge ones

    This is a simplified version of the Rabin fingerprint approach
    used by real systems like rsync, Perkeep, and Dropbox.
    """
    # mask determines average chunk size (works cleanly with powers of 2)
    mask = avg_size - 1

    chunks = []
    start = 0
    hash_val = 0

    for i in range(len(data)):
        # Update the rolling hash \u2014 each new byte shifts and mixes in
        hash_val = ((hash_val << 1) + data[i]) & 0xFFFFFFFF

        chunk_len = i - start + 1

        # Don't create a boundary if the chunk is too small
        if chunk_len < min_size:
            continue

        # Force a boundary if chunk is too large,
        # or create one if the hash condition is met
        if chunk_len >= max_size or (hash_val & mask) == 0:
            chunks.append(data[start:i + 1])
            start = i + 1
            hash_val = 0

    # Don't forget the last piece
    if start < len(data):
        chunks.append(data[start:])

    return chunks


print('\u2705 Both chunkers defined')
print()

# Quick test with sample data
sample = os.urandom(64 * 1024)  # 64 KB
fixed = fixed_size_chunks(sample)
content_defined = cdc_chunks(sample)

print(f'64 KB of random data:')
print(f'  Fixed-size: {len(fixed)} chunks (all exactly {8192:,} bytes)')
sizes = [len(c) for c in content_defined]
print(f'  CDC:        {len(content_defined)} chunks (sizes range: {min(sizes):,}\u2013{max(sizes):,} bytes)')

In [ ]:
# === Compare fixed-size vs CDC after a small edit ===

# Create original content (64 KB of repeating pattern \u2014 simulates real data)
original = bytes(range(256)) * 256  # 65,536 bytes

# Make a small edit: insert 100 bytes near the beginning
edit_pos = 100
insertion = b'INSERTED_CONTENT_HERE_' * 5  # 110 bytes
modified = original[:edit_pos] + insertion + original[edit_pos:]

print(f'Original file: {len(original):,} bytes')
print(f'Modified file: {len(modified):,} bytes (inserted {len(insertion)} bytes at position {edit_pos})')
print()

# --- Fixed-Size Chunking ---
orig_fixed = fixed_size_chunks(original)
mod_fixed  = fixed_size_chunks(modified)

orig_fixed_fps = set(sha256_hex(c) for c in orig_fixed)
mod_fixed_fps  = set(sha256_hex(c) for c in mod_fixed)
fixed_shared   = orig_fixed_fps & mod_fixed_fps

print(f'\U0001f4e6 Fixed-Size Chunking (chunk_size={8192})')
print(f'   Original chunks: {len(orig_fixed)}')
print(f'   Modified chunks: {len(mod_fixed)}')
print(f'   Shared (reusable): {len(fixed_shared)}')
print(f'   New chunks needed: {len(mod_fixed_fps - fixed_shared)}')
fixed_ratio = len(fixed_shared) / len(mod_fixed_fps) * 100 if mod_fixed_fps else 0
print(f'   Dedup ratio: {fixed_ratio:.1f}%')
print()

# --- Content-Defined Chunking ---
orig_cdc = cdc_chunks(original)
mod_cdc  = cdc_chunks(modified)

orig_cdc_fps = set(sha256_hex(c) for c in orig_cdc)
mod_cdc_fps  = set(sha256_hex(c) for c in mod_cdc)
cdc_shared   = orig_cdc_fps & mod_cdc_fps

print(f'\U0001f52c Content-Defined Chunking (avg_size={8192})')
print(f'   Original chunks: {len(orig_cdc)}')
print(f'   Modified chunks: {len(mod_cdc)}')
print(f'   Shared (reusable): {len(cdc_shared)}')
print(f'   New chunks needed: {len(mod_cdc_fps - cdc_shared)}')
cdc_ratio = len(cdc_shared) / len(mod_cdc_fps) * 100 if mod_cdc_fps else 0
print(f'   Dedup ratio: {cdc_ratio:.1f}%')
print()

# Summary
print('=' * 50)
print('\U0001f4ca COMPARISON SUMMARY')
print('=' * 50)
print(f'After inserting {len(insertion)} bytes at position {edit_pos}:')
print(f'  Fixed-size: {len(fixed_shared)}/{len(mod_fixed_fps)} chunks reused ({fixed_ratio:.1f}% dedup)')
print(f'  CDC:        {len(cdc_shared)}/{len(mod_cdc_fps)} chunks reused ({cdc_ratio:.1f}% dedup)')
print()
if len(cdc_shared) > len(fixed_shared):
    print('\u2705 CDC wins! Content-defined boundaries survive insertions much better.')
else:
    print('\u2139\ufe0f  Results depend on data patterns. CDC typically wins with real-world edits.')

## 📊 Storage Savings Calculator

Now let’s query our database to measure the actual savings from our dedup experiments.

**Key metrics:**
- **Logical bytes** — Total size of all files users *think* they have (sum of all file sizes)
- **Physical bytes** — Actual unique bytes stored in MinIO (count each chunk fingerprint once)
- **Dedup ratio** — `1 - (physical / logical)` — higher means more savings

In [ ]:
conn = get_db()
cur = conn.cursor()

# Total logical bytes (what users see \u2014 sum of all file sizes)
cur.execute("SELECT COALESCE(SUM(file_size), 0) FROM files WHERE status = 'uploaded'")
logical_bytes = cur.fetchone()[0]

# Total physical bytes (unique chunks actually stored)
# We count each unique fingerprint only once
cur.execute('''
    SELECT COALESCE(SUM(chunk_size), 0)
    FROM (
        SELECT DISTINCT ON (fingerprint) fingerprint, chunk_size
        FROM chunks
        WHERE status = 'uploaded'
    ) unique_chunks
''')
physical_bytes = cur.fetchone()[0]

# Also count file-level dedup savings
cur.execute('''
    SELECT COALESCE(SUM(file_size), 0)
    FROM (
        SELECT DISTINCT ON (fingerprint) fingerprint, file_size
        FROM files
        WHERE status = 'uploaded' AND fingerprint IS NOT NULL
    ) unique_files
''')
physical_file_level = cur.fetchone()[0]

# Stats
cur.execute("SELECT COUNT(*) FROM files WHERE status = 'uploaded'")
total_files = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM chunks WHERE status = 'uploaded'")
total_chunks = cur.fetchone()[0]
cur.execute("SELECT COUNT(DISTINCT fingerprint) FROM chunks WHERE status = 'uploaded'")
unique_chunks = cur.fetchone()[0]

conn.close()

print('=' * 55)
print('   \U0001f4ca DEDUPLICATION STORAGE REPORT')
print('=' * 55)
print()
print(f'  Files in database:         {total_files}')
print(f'  Total chunk records:       {total_chunks}')
print(f'  Unique chunk fingerprints: {unique_chunks}')
print(f'  Duplicate chunks:          {total_chunks - unique_chunks}')
print()
print(f'  Logical size  (users see): {logical_bytes:>12,} bytes')
print(f'  Physical size (stored):    {physical_bytes:>12,} bytes')
saved = logical_bytes - physical_bytes
ratio = (saved / logical_bytes * 100) if logical_bytes > 0 else 0
print(f'  Space saved:               {saved:>12,} bytes')
print(f'  Dedup ratio:               {ratio:>11.1f}%')
print()

# Simple bar chart
bar_width = 40
if logical_bytes > 0:
    physical_bar = max(1, int((physical_bytes / logical_bytes) * bar_width))
    saved_bar = bar_width - physical_bar
    print('  Storage visualization:')
    print(f'  Logical:  [{chr(9608) * bar_width}] {logical_bytes:,} bytes')
    print(f'  Physical: [{chr(9608) * physical_bar}{chr(9617) * saved_bar}] {physical_bytes:,} bytes')
    print(f'            {"":>{physical_bar + 12}}\u2191 saved!')
else:
    print('  (No data yet \u2014 run the cells above first)')

## 🔢 Reference Counting

There’s a tricky problem with dedup: **when can we safely delete data?**

### The Problem

If Alice and Bob both have files that share chunk `abc123`, and Alice deletes
her file, we **can’t** delete chunk `abc123` from MinIO — Bob still needs it!

### The Solution: Reference Counting

For each unique chunk, count how many files reference it. Only delete from
storage when the count drops to **zero**.

```
chunk "abc123":
  - Referenced by Alice's file  ─┐
  - Referenced by Bob's file    ─┤→ count = 2
                                 │
Alice deletes her file → count = 1 → DON'T delete from MinIO
Bob deletes his file   → count = 0 → SAFE to delete from MinIO ✅
```

Let’s implement this:

In [ ]:
def get_chunk_refcounts(cur):
    """Get reference counts for all chunk fingerprints."""
    cur.execute('''
        SELECT c.fingerprint, COUNT(*) as ref_count, c.chunk_size, c.storage_key
        FROM chunks c
        JOIN files f ON c.file_id = f.id
        WHERE c.status = 'uploaded' AND f.status = 'uploaded'
        GROUP BY c.fingerprint, c.chunk_size, c.storage_key
        ORDER BY ref_count DESC
    ''')
    return cur.fetchall()


def safe_delete_file(file_id):
    """Delete a file, only removing chunks from MinIO when refcount drops to 0.

    This is the safe way to delete in a deduplicated storage system:
    1. Look at each chunk the file owns
    2. Count how many OTHER files reference that chunk
    3. If no other file needs it \u2192 delete from MinIO
    4. If others still need it \u2192 leave it in MinIO
    5. Always delete the DB records for this file
    """
    conn = get_db()
    cur = conn.cursor()
    s3 = get_s3()

    # Get the file's chunks
    cur.execute(
        'SELECT fingerprint, storage_key FROM chunks WHERE file_id = %s',
        (file_id,),
    )
    file_chunks = cur.fetchall()

    print(f'\U0001f5d1\ufe0f  Deleting file {file_id} ({len(file_chunks)} chunks)...')
    chunks_freed = 0

    for fp, storage_key in file_chunks:
        # How many OTHER files reference this chunk?
        cur.execute(
            '''SELECT COUNT(DISTINCT file_id) FROM chunks
            WHERE fingerprint = %s AND file_id != %s AND status = 'uploaded' ''',
            (fp, file_id),
        )
        other_refs = cur.fetchone()[0]

        if other_refs == 0:
            # No other files need this chunk \u2014 safe to delete!
            s3.delete_object(Bucket=BUCKET, Key=storage_key)
            print(f'   \U0001f5d1\ufe0f  Chunk {fp[:12]}... \u2192 refcount=0 \u2192 DELETED from MinIO')
            chunks_freed += 1
        else:
            print(f'   \U0001f512 Chunk {fp[:12]}... \u2192 refcount={other_refs} \u2192 kept in MinIO')

    # Delete the file's chunk records and file record from the database
    cur.execute('DELETE FROM chunks WHERE file_id = %s', (file_id,))
    cur.execute('DELETE FROM files WHERE id = %s', (file_id,))
    conn.close()

    return chunks_freed


# === Show reference counts BEFORE deletion ===
conn = get_db()
cur = conn.cursor()

print('\U0001f4ca Chunk reference counts BEFORE deletion:')
print('-' * 65)
refcounts = get_chunk_refcounts(cur)
for fp, count, size, key in refcounts:
    status = '\u26a0\ufe0f  shared' if count > 1 else '   unique'
    print(f'   {fp[:16]}... | refs: {count} | size: {size:>6,} bytes | {status}')

conn.close()

In [ ]:
# === Delete Alice's chunked file and watch reference counting in action ===

print(f'Deleting Alice\'s chunked file (ID: {file_a_id})...')
print()
freed = safe_delete_file(file_a_id)

# Show reference counts AFTER deletion
print()
conn = get_db()
cur = conn.cursor()

print('\U0001f4ca Chunk reference counts AFTER deleting Alice\'s file:')
print('-' * 65)
refcounts = get_chunk_refcounts(cur)
if refcounts:
    for fp, count, size, key in refcounts:
        print(f'   {fp[:16]}... | refs: {count} | size: {size:>6,} bytes')
else:
    print('   (no chunks remaining)')

# Verify MinIO state
s3 = get_s3()
response = s3.list_objects_v2(Bucket=BUCKET)
obj_count = response.get('KeyCount', 0)

print(f'\n\U0001f4be Objects remaining in MinIO: {obj_count}')
print(f'   Chunks freed from storage:  {freed}')
print()
print('\u2705 Reference counting ensured we only deleted chunks that nobody else needs!')
conn.close()

## 🧹 Cleanup

Run this cell to remove all demo data created by this notebook.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Delete all demo data (chunks first due to foreign key constraints)
cur.execute('DELETE FROM chunks')
cur.execute('DELETE FROM files')
print('\U0001f5d1\ufe0f  Cleared database tables (files, chunks)')

# Remove all objects from MinIO bucket
s3 = get_s3()
response = s3.list_objects_v2(Bucket=BUCKET)
deleted = 0
if 'Contents' in response:
    for obj in response['Contents']:
        s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
        deleted += 1
print(f'\U0001f5d1\ufe0f  Removed {deleted} object(s) from MinIO')

conn.close()
print('\n\u2705 All clean! Ready for the next notebook.')

## 📝 Summary

### Key Takeaways

1. **🔐 File-level dedup** is the simplest approach — hash the whole file with SHA-256
   and skip uploads when the fingerprint already exists. Great for exact duplicates.

2. **🧩 Chunk-level dedup** breaks files into pieces and deduplicates each chunk
   independently. This catches partial overlaps — even files that are only *mostly*
   the same can share storage.

3. **✂️ Content-Defined Chunking (CDC)** picks chunk boundaries based on content
   (via a rolling hash), not position. This means small edits only affect nearby
   chunks, preserving dedup across file versions. Fixed-size chunking fails here
   because inserting a single byte shifts all boundaries.

4. **📊 Storage savings** add up fast. Dropbox saves 25%+ across billions of files.
   Our demo showed exactly how to measure logical vs. physical storage.

5. **🔢 Reference counting** is essential — when multiple files share a chunk, you
   must track how many references exist before deleting anything from storage.

### What’s Next?

In **Notebook 4**, we’ll explore **sync and conflict resolution** — how Dropbox keeps
files in sync across multiple devices and handles the tricky case when two people
edit the same file at the same time.